# Raw per-window regression outputs: sigma & chirp-mass recovery vs time-to-merger

Reads `raw_sig.csv` / `raw_bkg.csv` (from `save_outputs.py`) and plots, as a
function of the kernel position relative to the merger:

- **Fig 1** — predicted `sigma` (median line + 68% band), sig vs bkg.
- **Fig 2** — chirp-mass relative error `(pred - true)/true` on sig (median + 68% band).

x-axis = `kernel_right_s`: the 4 s kernel's right edge relative to coalescence (merger at 0).
So `-4` means the kernel ends 4 s before merger (spans `[-8,-4]`); `0` means the merger sits
at the kernel's right edge.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

OUTDIR = "/n/holystore01/LABS/iaifi_lab/Lab/kyoon/MODEL/aframe/reg-dev-latest/raw_csv/merger_4s_id2"  # noqa: E501
MIN_SNR = 0.0  # raise (e.g. 15, 30) to look at louder signals only

sig = pd.read_csv(f"{OUTDIR}/raw_sig.csv")
bkg = pd.read_csv(f"{OUTDIR}/raw_bkg.csv")
if MIN_SNR > 0:
    sig = sig[sig.snr >= MIN_SNR]
    bkg = bkg[bkg.snr >= MIN_SNR]
sig["rel_err"] = (
    sig.pred_chirp_mass - sig.true_chirp_mass_src
) / sig.true_chirp_mass_src
print(f"strains={sig.strain_id.nunique()}")


def band(df, col):
    """median + 16/84 percentile vs kernel_right_s."""
    g = df.groupby("kernel_right_s")[col]
    return g.median(), g.quantile(0.16), g.quantile(0.84)

In [ ]:
# Fig 1 -- sigma vs time-to-merger (sig vs bkg)
fig, ax = plt.subplots(figsize=(8, 5))
for df, name, c in [
    (sig, "injected (sig)", "C0"),
    (bkg, "background (bkg)", "C1"),
]:
    med, lo, hi = band(df, "pred_sigma")
    ax.plot(med.index, med.values, color=c, lw=2, label=f"{name} median")
    ax.fill_between(
        med.index,
        lo.values,
        hi.values,
        color=c,
        alpha=0.25,
        label=f"{name} 68%",
    )
ax.set_xlabel("kernel right edge relative to merger [s]")
ax.set_ylabel(r"predicted $\sigma_{\mathcal{M}}$")
ax.set_title(f"Predicted sigma vs time-to-merger (SNR>={MIN_SNR:g})")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
# Fig 2 -- chirp-mass relative error vs time-to-merger (sig)
fig, ax = plt.subplots(figsize=(8, 5))
med, lo, hi = band(sig, "rel_err")
ax.plot(med.index, med.values, color="C2", lw=2, label="median")
ax.fill_between(
    med.index, lo.values, hi.values, color="C2", alpha=0.25, label="68%"
)
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("kernel right edge relative to merger [s]")
ax.set_ylabel(
    r"chirp-mass relative error $(\hat{\mathcal{M}}-\mathcal{M})/\mathcal{M}$"
)
ax.set_title(f"Chirp-mass recovery vs time-to-merger (SNR>={MIN_SNR:g})")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()